In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/MP-PCA
!ls

/content/drive/MyDrive/MP-PCA
add_noise.ipynb  localpca_dn.py     noisy_data	 Real_Imag_Combine.ipynb
denoised	 meas_gre_dir1.mat  __pycache__  시각화.ipynb


In [3]:
import scipy.io as sio
import numpy as np

mat = sio.loadmat("meas_gre_dir1.mat")

print(mat.keys())

dict_keys(['__header__', '__version__', '__globals__', 'B0_dir', 'CF', 'delta_TE', 'mask_brain', 'meas_gre', 'te_gre', 'tolvalue', 'voxel_size_gre'])


In [4]:
meas_gre = mat["meas_gre"]

print("shape :", meas_gre.shape)
print("dtype :", meas_gre.dtype)
print("complex :", np.iscomplexobj(meas_gre))

shape : (256, 224, 176, 6)
dtype : complex128
complex : True


In [5]:
import numpy as np

def add_complex_gaussian_noise(
    real,
    imag,
    mask,
    noise_level,
    seed=42
):
    rng = np.random.default_rng(seed)

    # 원본 magnitude
    magnitude = np.sqrt(real**2 + imag**2)

    # brain mask 내부 평균 magnitude
    mean_mag = magnitude[mask].mean()

    # noise sigma
    sigma = noise_level * mean_mag

    # noise 생성
    noise_real = rng.normal(
        0.0,
        sigma,
        size=real.shape
    ).astype(np.float32)

    noise_imag = rng.normal(
        0.0,
        sigma,
        size=imag.shape
    ).astype(np.float32)

    # 원본 복사
    noisy_real = real.copy().astype(np.float32)
    noisy_imag = imag.copy().astype(np.float32)

    # mask 내부에만 noise 추가
    noisy_real[mask] += noise_real[mask]
    noisy_imag[mask] += noise_imag[mask]

    noisy_complex = noisy_real + 1j * noisy_imag

    return noisy_real, noisy_imag, noisy_complex, sigma

In [6]:
real = np.real(meas_gre).astype(np.float32)
imag = np.imag(meas_gre).astype(np.float32)

magnitude = np.abs(meas_gre)

mask_brain = magnitude[..., 0] > 0

print(real.shape)
print(imag.shape)
print(mask_brain.shape)

(256, 224, 176, 6)
(256, 224, 176, 6)
(256, 224, 176)


In [7]:
import scipy.io as sio
import os

noise_levels = [0.1, 0.3, 0.5]

save_dir = "noisy_data"
os.makedirs(save_dir, exist_ok=True)

for level in noise_levels:

    percent = int(level * 100)

    noisy_real, noisy_imag, noisy_complex, sigma = add_complex_gaussian_noise(
        real,
        imag,
        mask_brain,
        noise_level=level,
        seed=42
    )

    save_path = os.path.join(
        save_dir,
        f"noisy_meas_gre_dir1_{percent}.mat"
    )

    sio.savemat(
        save_path,
        {
            "noisy_real": noisy_real,
            "noisy_imag": noisy_imag,
            "sigma": sigma
        }
    )

    print(
        f"{percent}% 완료 | "
        f"sigma = {sigma:.4f} | "
        f"{save_path}"
    )

10% 완료 | sigma = 0.3049 | noisy_data/noisy_meas_gre_dir1_10.mat
30% 완료 | sigma = 0.9146 | noisy_data/noisy_meas_gre_dir1_30.mat
50% 완료 | sigma = 1.5243 | noisy_data/noisy_meas_gre_dir1_50.mat


In [8]:
!ls -lh noisy_data

total 1.4G
-rw------- 1 root root 463M Oct 12 06:24 noisy_meas_gre_dir1_10.mat
-rw------- 1 root root 463M Oct 12 06:24 noisy_meas_gre_dir1_30.mat
-rw------- 1 root root 463M Oct 12 06:24 noisy_meas_gre_dir1_50.mat


In [9]:
# mask 밖에서는 원본과 완전히 같아야 함
print(
    "Real mask 밖 최대 차이:",
    np.max(np.abs(noisy_real[~mask_brain] - real[~mask_brain]))
)

print(
    "Imag mask 밖 최대 차이:",
    np.max(np.abs(noisy_imag[~mask_brain] - imag[~mask_brain]))
)

Real mask 밖 최대 차이: 0.0
Imag mask 밖 최대 차이: 0.0
